In [ ]:
%pip install -q langchain
%pip install -q langchain-core
%pip install -q langchain-classic
%pip install -q langchain-community
%pip install -q langchain-openai
%pip install -q langchain-huggingface
%pip install -q langchain-chroma
%pip install -q langchain-pinecone

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-5.4-nano")
answer = llm.invoke("컴퓨터 과학의 아버지는?")
answer.content

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
    다음 질문에 한 문장으로 답하세요.
    [질문] {query}
""")
chain = prompt | llm
answer = chain.invoke({"query": "컴퓨터 과학의 아버지는?"})
answer.content

In [ ]:
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()
chain = prompt | llm | output_parser
chain.invoke({"query": "컴퓨터 과학의 아버지는?"})

In [ ]:
from langchain_core.runnables import RunnablePassthrough
preprocess = RunnablePassthrough.assign(
    context=lambda x: "컴퓨터 과학의 아버지는 주니온 교수입니다."
)
prompt = ChatPromptTemplate.from_template("""
    다음 문맥만 참고하여 질문에 한 문장으로 답하세요.
    [문맥] {context}
    [질문] {query}
""")
chain = preprocess | prompt | llm | output_parser
chain.invoke({"query": "컴퓨터 과학의 아버지는?"})

In [ ]:
from langchain_core.runnables import RunnableLambda
chain = (
    ChatPromptTemplate.from_template("한 문장으로 질문에 답하세요. [질문] {question}")
    | llm
    | output_parser
    | RunnableLambda(lambda output: {"answer": output})
    | ChatPromptTemplate.from_template("이 답변을 영어로 번역하세요. [답변] {answer}")
    | llm
    | output_parser
)
chain.invoke({"question": "컴퓨터 과학의 아버지는?"})

In [ ]:
from langchain_core.runnables import RunnableParallel

another_chain = (
    ChatPromptTemplate.from_template("이 질문을 영어로 번역하세요. [질문] {question}")
    | llm
    | output_parser
)

format_parser = RunnableLambda(
    lambda x: f"[질문]\n{x['question']}\n\n[답변]\n{x['answer']}"
)
parallel_chain = (
    RunnableParallel(question=another_chain, answer=chain)
    | format_parser
)
print(parallel_chain.invoke({"question": "컴퓨터 과학의 아버지는?"}))

In [ ]:
chain = (
    ChatPromptTemplate.from_template("이 질문에 답하세요. [질문] {question}")
    | ChatOpenAI(model="gpt-5.4-nano")
    | StrOutputParser()
)
for i, chunk in enumerate(chain.stream({"question": "컴퓨터 과학의 아버지는?"})):
    print(i, chunk)

In [ ]:
from langchain_core.documents import Document
documents = [
    Document(page_content="경북대학교 KDT 14기의 교육 장소는 경일대학교이다."),
    Document(page_content="주니온 교수는 전이학습 모델, 웹 기반, 클라우드 기반 AI 서비스를 강의한다."),
    Document(page_content="주니온 교수는 KDT 14기의 교육을 담당한다.")
]

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
llm = ChatOpenAI(model="gpt-5.4-nano")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

In [ ]:
docs = retriever.invoke("교육 위치는?")
for doc in docs:
    print(doc.page_content)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [ ]:
docs = retriever.invoke("주니온 교수는?")
for doc in docs:
    print(doc.page_content)

In [ ]:
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

In [ ]:
docs = retriever.invoke("주니온 교수가 담당하는 강의는?")
context = format_docs(docs)
print(context)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
    다음 문맥만 사용하여 질문에 한 문장으로 답하세요.
    문맥에서 알 수 없는 내용은 모른다고 답하세요.
    [문맥] {context}
    [질문] {question}
""")

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)
rag_chain.invoke("주니온 교수가 강의하는 과목은?")

In [ ]:
question = "주니온 교수가 강의하는 장소는?"
context = format_docs(retriever.invoke(question))
answer = rag_chain.invoke(question)
print(f"[문맥]\n{context}\n\n[답변]\n{answer}")

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
context = format_docs(retriever.invoke(question))
answer = rag_chain.invoke(question)
print(f"[문맥]\n{context}\n\n[답변]\n{answer}")

In [ ]:
reasoning_prompt = ChatPromptTemplate.from_template("""
    다음 문맥을 근거로 질문에 답하세요.
    문맥에 있는 내용을 연결하여 알 수 있는 내용은 답하세요.
    문맥의 내용을 추론해도 알 수 없는 내용은 모른다고 답하세요.
    [문맥] {context}
    [질문] {question}
""")
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | reasoning_prompt | llm | StrOutputParser()
)
rag_chain.invoke(question)

### Upstage Solar Pro 3 모델 사용하기

In [ ]:
%pip install -q langchain-upstage

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
key = os.getenv("UPSTAGE_API_KEY")
key[:10]

In [ ]:
from langchain_upstage import ChatUpstage

llm = ChatUpstage(model="solar-pro3")
answer = llm.invoke("컴퓨터 과학의 아버지는?")
answer.content